In [24]:
import os
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset

In [25]:
class SegmentationDataset(Dataset):
    def __init__(self, auth_img_dir, forged_img_dir, forged_mask_dir, target_size = (256, 320)):
        self.samples = []
        self.target_size = target_size

        for img in os.listdir(auth_img_dir):
            if img.lower().endswith((".jpg", ".png", ".jpeg")):
                self.samples.append({
                    "image_path": os.path.join(auth_img_dir, img),
                    "mask_path": None,
                    "type": "authentic"
                })

        for img in os.listdir(forged_img_dir):
            if img.lower().endswith((".jpg", ".png", ".jpeg")):
                self.samples.append({
                    "image_path": os.path.join(forged_img_dir, img),
                    "mask_path": os.path.join(
                        forged_mask_dir,
                        os.path.splitext(img)[0] + ".npy"
                    ),
                    "type": "forged"
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        image = Image.open(sample["image_path"]).convert("RGB")
        image = image.resize((self.target_size[1], self.target_size[0]))
        image = np.array(image, dtype=np.float32) / 255.0

        height, width, _ = image.shape

        if sample["type"] == "authentic":
            mask = np.zeros((height, width), dtype=np.float32)

        else:
            mask = np.load(sample["mask_path"]).astype(np.float32)

            if mask.ndim == 3:
                mask = mask[0]

            mask = (mask > 0).astype(np.float32)

            mask = Image.fromarray(mask)
            mask = mask.resize((self.target_size[1], self.target_size[0]), Image.NEAREST)
            mask = np.array(mask, dtype = np.float32)


        image = np.transpose(image, (2, 0, 1))   # C × H × W
        mask = np.expand_dims(mask, axis=0)      # 1 × H × W

        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)

        return image, mask



In [26]:
auth_img_dir = "./data/train_images/authentic"
forged_img_dir = "./data/train_images/forged"
forged_mask_dir = "./data/train_masks"


dataset = SegmentationDataset(auth_img_dir, forged_img_dir, forged_mask_dir )

In [27]:
from torch.utils.data import random_split, DataLoader

dataset_size = len(dataset)
val_size = int(0.2 * dataset_size)
train_size = dataset_size - val_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [28]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2
import torch.nn.functional as F


class MobileNetV2_Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = mobilenet_v2(weights="IMAGENET1K_V1")
        self.features = backbone.features

    def forward(self, x):
        outputs = []

        for i, layer in enumerate(self.features):
            x = layer(x)

            # These indices correspond to resolution drops
            if i == 3:   # ~80x64
                T3 = x
            elif i == 6: # ~40x32
                T4 = x
            elif i == 13: # ~20x16
                T5 = x

        return T3, T4, T5

In [29]:
x = torch.randn(1,3,320,256)
encoder = MobileNetV2_Encoder()
T3,T4,T5 = encoder(x)

print(T3.shape)  # expect ~ (1, 24, 80,64)
print(T4.shape)  # expect ~ (1, 32, 40,32)
print(T5.shape)  # expect ~ (1, 96, 20,16)

torch.Size([1, 24, 80, 64])
torch.Size([1, 32, 40, 32])
torch.Size([1, 96, 20, 16])


In [30]:
class CorrelationModule(nn.Module):
    def __init__(self, gamma=4, topk=5):
        super().__init__()
        self.gamma = gamma
        self.topk = topk

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W

        # reshape to (B, C, N)
        Fmat = x.view(B, C, N)

        # normalize for cosine similarity
        Fmat = F.normalize(Fmat, dim=1)

        # compute affinity matrix A_hat = F^T F
        A_hat = torch.bmm(Fmat.transpose(1,2), Fmat)  # (B, N, N)

        # build suppression matrix Phi
        device = x.device

        coords = torch.stack(torch.meshgrid(
            torch.arange(H, device=device),
            torch.arange(W, device=device),
            indexing='ij'
        ), dim=-1).view(-1, 2)  # (N,2)

        diff = coords.unsqueeze(1) - coords.unsqueeze(0)
        dist2 = (diff[...,0]**2 + diff[...,1]**2).float()

        Phi = 1 - torch.exp(-dist2 / (2 * self.gamma**2))
        Phi = Phi.unsqueeze(0)  # (1,N,N)

        # suppress diagonal similarity
        A = A_hat * Phi

        # select top-k similarity per location
        topk_vals, _ = torch.topk(A, k=self.topk, dim=-1)

        # average top-k scores
        K = topk_vals.mean(dim=-1)  # (B,N)

        # reshape back to spatial map
        K = K.view(B, 1, H, W)

        return K

In [31]:
cor = CorrelationModule()
K3 = cor(T3)
print(K3.shape)


torch.Size([1, 1, 80, 64])


In [32]:
class VRSA(nn.Module):
    def __init__(self):
        super().__init__()

        # ASPP with 1 input channel
        self.aspp1 = nn.Conv2d(1, 1, 3, padding=4, dilation=4)
        self.aspp2 = nn.Conv2d(1, 1, 3, padding=8, dilation=8)
        self.aspp3 = nn.Conv2d(1, 1, 3, padding=12, dilation=12)
        self.aspp4 = nn.Conv2d(1, 1, 3, padding=16, dilation=16)

        self.project = nn.Conv2d(4, 1, 1)

        # Spatial attention
        self.spatial_att = nn.Sequential(
            nn.Conv2d(1, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, x):
        a1 = self.aspp1(x)
        a2 = self.aspp2(x)
        a3 = self.aspp3(x)
        a4 = self.aspp4(x)

        out = torch.cat([a1,a2,a3,a4], dim=1)
        out = self.project(out)

        att = self.spatial_att(out)
        out = out + out * att

        return out

In [33]:
vrsa = VRSA()
refined = vrsa(K3)
print(refined.shape)

torch.Size([1, 1, 80, 64])


In [34]:
class IRB(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        hidden = in_ch * 2

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU6(inplace=True),

            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU6(inplace=True),

            nn.Conv2d(hidden, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        )

    def forward(self, x):
        return self.block(x)

In [35]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.irb4 = IRB(1,1)
        self.irb3 = IRB(1,1)

        self.final = nn.Conv2d(1,1,1)

    def forward(self, K3, K4, K5):

        # 20x16 → 40x32
        x = F.interpolate(K5, scale_factor=2, mode='bilinear', align_corners=False)
        x = x + K4
        x = self.irb4(x)

        # 40x32 → 80x64
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = x + K3
        x = self.irb3(x)

        # 80x64 → 320x256
        x = F.interpolate(x, size=(256,320), mode='bilinear', align_corners=False)

        out = self.final(x)

        return out

In [36]:
decoder = Decoder()

# fake test tensors
K5 = torch.randn(1,1,20,16)
K4 = torch.randn(1,1,40,32)
K3 = torch.randn(1,1,80,64)

out = decoder(K3,K4,K5)

print(out.shape)

torch.Size([1, 1, 256, 320])


In [37]:
class CMSegNet_Exact(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = MobileNetV2_Encoder()

        # Correlation modules
        self.cor3 = CorrelationModule()
        self.cor4 = CorrelationModule()
        self.cor5 = CorrelationModule()

        # VRSA modules
        self.vrsa3 = VRSA()
        self.vrsa4 = VRSA()
        self.vrsa5 = VRSA()

        # Decoder
        self.decoder = Decoder()

    def forward(self, x):
        # Encoder
        T3, T4, T5 = self.encoder(x)

        # Correlation + VRSA at each level
        K3 = self.vrsa3(self.cor3(T3))   # (B,1,80,64)
        K4 = self.vrsa4(self.cor4(T4))   # (B,1,40,32)
        K5 = self.vrsa5(self.cor5(T5))   # (B,1,20,16)

        # Decoder fusion
        out = self.decoder(K3, K4, K5)

        return out

In [38]:
model = CMSegNet_Exact()
x = torch.randn(1,3,320,256)
y = model(x)
print(y.shape)

torch.Size([1, 1, 256, 320])


In [39]:
class ConditionalDiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)

        B = probs.shape[0]
        losses = []

        for i in range(B):
            p = probs[i]
            y = targets[i]

            y_sum = y.sum()

            if y_sum > 0:
                intersection = (p * y).sum()
                union = p.sum() + y_sum
                dice = (2 * intersection + self.eps) / (union + self.eps)
                loss = 1 - dice
            
            else:
                loss = p.mean()

            losses.append(loss)
        
        return torch.stack(losses).mean()

In [40]:
device = "cuda"
model = CMSegNet_Exact().to(device)
criterion = ConditionalDiceLoss()

x = torch.randn(1,3,256,320).to(device)
mask = torch.randint(0,2,(1,1,256,320)).float().to(device)

out = model(x)
loss = criterion(out, mask)
loss.backward()

print("Loss:", loss.item())
print("Backward pass successful.")

Loss: 0.57700115442276
Backward pass successful.


In [41]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CMSegNet_Exact().to(device)
criterion = ConditionalDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [42]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, masks)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


def validate_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks  = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            running_loss += loss.item()

    return running_loss / len(loader)

In [20]:
num_epochs = 65
best_val_loss = float("inf")

for epoch in range(num_epochs):

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss   = validate_one_epoch(model, val_loader, criterion)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_cmseg_model.pth")
        print("  → Best model saved.")

print("5 Epoch sanity training done.")

Epoch [1/65] | Train Loss: 0.7221 | Val Loss: 0.6997
  → Best model saved.
Epoch [2/65] | Train Loss: 0.7088 | Val Loss: 0.6850
  → Best model saved.
Epoch [3/65] | Train Loss: 0.6931 | Val Loss: 0.6723
  → Best model saved.
Epoch [4/65] | Train Loss: 0.6739 | Val Loss: 0.6488
  → Best model saved.
Epoch [5/65] | Train Loss: 0.6550 | Val Loss: 0.6315
  → Best model saved.
Epoch [6/65] | Train Loss: 0.6334 | Val Loss: 0.6070
  → Best model saved.
Epoch [7/65] | Train Loss: 0.6068 | Val Loss: 0.5287
  → Best model saved.
Epoch [8/65] | Train Loss: 0.5804 | Val Loss: 0.5687
Epoch [9/65] | Train Loss: 0.5546 | Val Loss: 0.5010
  → Best model saved.
Epoch [10/65] | Train Loss: 0.5306 | Val Loss: 0.5086
Epoch [11/65] | Train Loss: 0.5077 | Val Loss: 0.4958
  → Best model saved.
Epoch [12/65] | Train Loss: 0.4862 | Val Loss: 0.4673
  → Best model saved.
Epoch [13/65] | Train Loss: 0.4682 | Val Loss: 0.4768
Epoch [14/65] | Train Loss: 0.4515 | Val Loss: 0.5399
Epoch [15/65] | Train Loss: 0.433

In [44]:
# Recreate model architecture first
model = CMSegNet_Exact().to(device)

# Load best weights
model.load_state_dict(torch.load("best_cmseg_model.pth", map_location=device))

model.eval()

print("Best model loaded successfully.")

Best model loaded successfully.


C:\Users\asuss\AppData\Local\Temp\ipykernel_22412\4089293627.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_cmseg_model.pth", map

In [45]:
import numpy as np
from sklearn.metrics import roc_auc_score


def hard_dice_score(prob_map, gt_mask, threshold=0.5, eps=1e-6):
    pred = (prob_map > threshold).float()
    intersection = (pred * gt_mask).sum()
    union = pred.sum() + gt_mask.sum()
    return (2 * intersection + eps) / (union + eps)


def hard_iou_score(prob_map, gt_mask, threshold=0.5, eps=1e-6):
    pred = (prob_map > threshold).float()
    intersection = (pred * gt_mask).sum()
    union = pred.sum() + gt_mask.sum() - intersection
    return (intersection + eps) / (union + eps)


def precision_score(prob_map, gt_mask, threshold=0.5, eps=1e-6):
    pred = (prob_map > threshold).float()
    tp = (pred * gt_mask).sum()
    fp = (pred * (1 - gt_mask)).sum()
    return (tp + eps) / (tp + fp + eps)


def recall_score(prob_map, gt_mask, threshold=0.5, eps=1e-6):
    pred = (prob_map > threshold).float()
    tp = (pred * gt_mask).sum()
    fn = ((1 - pred) * gt_mask).sum()
    return (tp + eps) / (tp + fn + eps)

In [46]:
@torch.no_grad()
def evaluate_full_metrics(model, loader, device, threshold=0.5):

    model.eval()

    dice_scores = []
    iou_scores = []
    precision_scores = []
    recall_scores = []
    auc_scores = []

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits)

        for i in range(images.size(0)):
            gt_mask = masks[i, 0]

            # evaluate only forged samples
            if gt_mask.sum() > 0:

                prob_map = probs[i, 0]

                dice = hard_dice_score(prob_map, gt_mask, threshold)
                iou  = hard_iou_score(prob_map, gt_mask, threshold)
                prec = precision_score(prob_map, gt_mask, threshold)
                rec  = recall_score(prob_map, gt_mask, threshold)

                # ROC-AUC (pixel level)
                try:
                    auc = roc_auc_score(
                        gt_mask.cpu().numpy().flatten(),
                        prob_map.cpu().numpy().flatten()
                    )
                except:
                    auc = 0.0

                dice_scores.append(dice.item())
                iou_scores.append(iou.item())
                precision_scores.append(prec.item())
                recall_scores.append(rec.item())
                auc_scores.append(auc)

    return {
        "mean_hard_dice": np.mean(dice_scores) if dice_scores else 0.0,
        "mean_hard_iou": np.mean(iou_scores) if iou_scores else 0.0,
        "mean_precision": np.mean(precision_scores) if precision_scores else 0.0,
        "mean_recall": np.mean(recall_scores) if recall_scores else 0.0,
        "mean_roc_auc": np.mean(auc_scores) if auc_scores else 0.0,
        "num_forged_samples": len(dice_scores)
    }

In [47]:
metrics = evaluate_full_metrics(
    model,
    val_loader,
    device,
    threshold=0.5
)

print(metrics)

{'mean_hard_dice': np.float64(0.46695254075259734), 'mean_hard_iou': np.float64(0.3600529543174146), 'mean_precision': np.float64(0.44457546353534794), 'mean_recall': np.float64(0.5930499944657183), 'mean_roc_auc': np.float64(0.8456753753454866), 'num_forged_samples': 564}
